## Calibration for the stereo camera

- To do: Needs, thresholding to make image white-black instead of gray
- v1.1: Detected most checkerboards but still horrible result
- v1.0: Detected multiple checkerboards

In [ ]:
## 
import cv2
import glob
import numpy as np

def find_all_checkerboards_with_masks_enhanced(image, checkerboards_dims):
    detected_corners = []
    mask = np.ones_like(image, dtype=np.uint8)  # Initialize mask

    for checkerboard_dims in checkerboards_dims:
        while True:
            masked_image = cv2.bitwise_and(image, image, mask=mask)

            # Pre-process the image to improve contrast
            enhanced_image = cv2.equalizeHist(masked_image)

            # Find checkerboards with improved flags
            ret, corners = cv2.findChessboardCorners(enhanced_image, checkerboard_dims, 
                                                     flags=cv2.CALIB_CB_ADAPTIVE_THRESH + 
                                                           cv2.CALIB_CB_NORMALIZE_IMAGE)

            if ret:
                # Refine corners for accuracy
                # criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
                # corners = cv2.cornerSubPix(enhanced_image, corners, (11, 11), (-1, -1), criteria)
                
                detected_corners.append((corners, checkerboard_dims))

                # Mask out the detected checkerboard region (use convex hull for better masking)
                polygon_points = np.array([corner[0] for corner in corners], dtype=np.int32)
                cv2.fillConvexPoly(mask, polygon_points, 0)
            else:
                break

    return detected_corners

def processStereoImages(left_image_path, right_image_path, checkerboards_dims: list, visualize=True):
    left_img = cv2.imread(left_image_path)
    right_img = cv2.imread(right_image_path)
    gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)

    all_corners_left = find_all_checkerboards_with_masks_enhanced(gray_left, checkerboards_dims)
    all_corners_right = find_all_checkerboards_with_masks_enhanced(gray_right, checkerboards_dims)

    if visualize:
        for corners, dims in all_corners_left:
            cv2.drawChessboardCorners(left_img, dims, corners, True)
        for corners, dims in all_corners_right:
            cv2.drawChessboardCorners(right_img, dims, corners, True)

        cv2.imshow("Left Image with All Corners", left_img)
        cv2.imshow("Right Image with All Corners", right_img)
        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cv2.destroyAllWindows()

    return left_img, right_img, all_corners_left, all_corners_right

def stereoCalibration(left_img_dir, right_img_dir, corners, visualize=True):
    left_image_files = glob.glob(left_img_dir + r"\*.png")
    right_image_files = glob.glob(right_img_dir + r"\*.png")

    # left_image_files = [left_image_files[-1]]
    # right_image_files = [right_image_files[-1]]

    left_images = []
    right_images = []
    corners_left = []
    corners_right = []

    for left_image_file, right_image_file in zip(left_image_files, right_image_files):
        l, r, cl, cr = processStereoImages(left_image_file, right_image_file, corners, visualize=True)
        left_images.append(l)
        right_images.append(r)
        corners_left.append(cl)
        corners_right.append(cr)

    if visualize:
        left_image = left_images[-1]
        cs_l = corners_left[-1]
        right_image = right_images[-1]
        cs_r = corners_right[-1]
        for c_l, dims_l in cs_l:
            cv2.drawChessboardCorners(left_image, dims_l, c_l, True)
        for c_r, dims_r in cs_r:
            cv2.drawChessboardCorners(right_image, dims_r, c_r, True)

        cv2.imshow("Final Left Image with All Corners", left_image)
        cv2.imshow("Final Right Image with All Corners", right_image)

        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cv2.destroyAllWindows()

# Example usage:
left_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (7,11), (7,11), 
    (5,7), (5,7), (5,7), (5,7), (5,7), 
    (7,5), (7,5), (7,5), (7,5), (7,5), 
    (15, 5)
]
# corners = [
#     (11, 7), (11, 7),
#     (7, 5), (7, 5), (7, 5), (7, 5), (7, 5),
#     (5, 7), (5, 7), (5, 7), (5, 7), (5, 7), 
#     (5, 15)
# ]
stereoCalibration(left_img_path, right_img_path, corners, visualize=True)


## Kitti dataset LSVM
- have to have the structure:
```plaintext
kitti_dataset/
├── data_object_image_2/        # Left camera images
│   ├── training/
│   │   ├── image_2/            # Left camera images
│   │   └── image_3/            # Right camera images (optional if downloaded)
├── data_object_label_2/        # Labels
│   ├── training/
│   │   └── label_2/            # 3D object labels
├── data_object_calib/          # Calibration files
│   ├── training/
│   │   └── calib/              # Calibration data
```
-

In [ ]:
import pykitty
import cv2
import os
import numpy as np
from scipy.io import loadmat

In [ ]:

# Define paths
left_image_path = 'kitti_dataset/images/training/image_2'
right_image_path = 'kitti_dataset/images/training/image_3'

# Load an example frame (e.g., 000000.png)
frame_id = '000000'

left_image = cv2.imread(os.path.join(left_image_path, frame_id + '.png'))
right_image = cv2.imread(os.path.join(right_image_path, frame_id + '.png'))

# Display images
cv2.imshow('Left Image', left_image)
cv2.imshow('Right Image', right_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
label_path = 'kitti_dataset/labels/training/label_2'

def read_labels(label_file):
    labels = []
    with open(label_file, 'r') as file:
        for line in file:
            parts = line.strip().split(' ')
            obj = {
                'type': parts[0],  # Car, Pedestrian, Cyclist
                'bbox': [float(p) for p in parts[4:8]],  # 2D bounding box in image (xmin, ymin, xmax, ymax)
                'dimensions': [float(p) for p in parts[8:11]],  # 3D dimensions (h, w, l)
                'location': [float(p) for p in parts[11:14]],  # 3D location (x, y, z)
                'rotation_y': float(parts[14]),  # Rotation around Y-axis
            }
            labels.append(obj)
    return labels

# Example usage
frame_label_file = os.path.join(label_path, frame_id + '.txt')
labels = read_labels(frame_label_file)
print(labels)


In [ ]:
calib_path = 'kitti_dataset/calib/training/calib'

def read_calib(calib_file):
    with open(calib_file, 'r') as file:
        lines = file.readlines()
        calib = {}
        for line in lines:
            key, value = line.split(':', 1)
            calib[key] = np.array([float(x) for x in value.strip().split()])
        return calib

# Example usage
frame_calib_file = os.path.join(calib_path, frame_id + '.txt')
calib = read_calib(frame_calib_file)
print(calib['P2'])  # Projection matrix for left camera


In [8]:
import importlib
import detect2  # Your script
importlib.reload(detect2)

<module 'detect2' from 'C:\\Users\\szakt\\Desktop\\DTU\\Perception\\FinalProject\\yolov7\\detect2.py'>

In [1]:
import sys
sys.path.append(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7_BS")
import torch
from detect2 import main
import matplotlib.pyplot as plt
import cv2
img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data"
model_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\yolov7.pt"
output_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\Output"
main()

C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7_BS\models\experimental.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(w, map_location=map_

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


c:\Users\szakt\Desktop\DTU\venv\Lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Images processed:   5%|▍         | 7/145 [00:02<00:57,  2.39it/s]


KeyboardInterrupt: 

In [5]:
import cv2
import os
from glob import glob
from tqdm import tqdm

def create_video_from_images(image_folder, output_video_path, fps=30):
    """
    Creates a video from a sequence of images.

    Args:
        image_folder (str): Path to the folder containing images.
        output_video_path (str): Path to save the output video file.
        fps (int, optional): Frames per second for the output video. Defaults to 30.
    """
    # Get list of image files in the folder, sorted by filename
    image_files = sorted(glob(os.path.join(image_folder, '*')), key=os.path.getmtime)
    image_files = [f for f in image_files if os.path.isfile(f) and f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not image_files:
        print("No images found in the specified folder.")
        return

    # Read the first image to get dimensions
    first_frame = cv2.imread(image_files[0])
    if first_frame is None:
        print(f"Failed to read image {image_files[0]}")
        return
    height, width, layers = first_frame.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # You can use 'XVID' or 'avc1' depending on your needs
    video = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Iterate over images and write them to the video
    for image_file in tqdm(image_files, "Processing images:"):
        frame = cv2.imread(image_file)
        if frame is None:
            print(f"Failed to read image {image_file}, skipping.")
            continue
        video.write(frame)

    video.release()
    print(f"Video saved to {output_video_path}")

# Create a video from the processed images
image_folder = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\Output"  # The folder where processed images are saved
output_video_path = os.path.join(image_folder,"output_video.mp4")
fps = 10  # Set frames per second for the video

create_video_from_images(image_folder, output_video_path, fps)


Processing images:: 100%|██████████| 114/114 [00:02<00:00, 46.74it/s]

Video saved to C:\Users\szakt\Desktop\DTU\Perception\FinalProject\Output\output_video.mp4


In [ ]:
import sys
import os
import torch
import cv2
import numpy as np
from pathlib import Path
import sys
sys.path.append(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7_BS")
from models.yolo import Model  # Import the YOLO model
import torchvision
from torch.nn import Module

# Helper functions
def xywh2xyxy(x):
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[:, 0] = x[:, 0] - x[:, 2] / 2  # top left x
    y[:, 1] = x[:, 1] - x[:, 3] / 2  # top left y
    y[:, 2] = x[:, 0] + x[:, 2] / 2  # bottom right x
    y[:, 3] = x[:, 1] + x[:, 3] / 2  # bottom right y
    return y

def clip_coords(boxes, img_shape):
    boxes[:, 0].clamp_(0, img_shape[1])  # x1
    boxes[:, 1].clamp_(0, img_shape[0])  # y1
    boxes[:, 2].clamp_(0, img_shape[1])  # x2
    boxes[:, 3].clamp_(0, img_shape[0])  # y2

def scale_coords(img1_shape, coords, img0_shape, ratio_pad=None):
    if ratio_pad is None:  # calculate from img0_shape
        gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])  # gain  = old / new
        pad = (img1_shape[1] - img0_shape[1] * gain) / 2, (img1_shape[0] - img0_shape[0] * gain) / 2  # wh padding
    else:
        gain = ratio_pad[0][0]
        pad = ratio_pad[1]

    coords[:, [0, 2]] -= pad[0]  # x padding
    coords[:, [1, 3]] -= pad[1]  # y padding
    coords[:, :4] /= gain
    clip_coords(coords, img0_shape)
    return coords

def non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45, classes=None):
    nc = prediction.shape[2] - 5  # number of classes
    xc = prediction[..., 4] > conf_thres  # candidates

    output = [torch.zeros((0, 6), device=prediction.device)] * prediction.shape[0]
    for xi, x in enumerate(prediction):
        x = x[xc[xi]]  # confidence
        if not x.shape[0]:
            continue

        x[:, 5:] *= x[:, 4:5]  # conf = obj_conf * cls_conf
        box = xywh2xyxy(x[:, :4])
        conf, j = x[:, 5:].max(1, keepdim=True)
        x = torch.cat((box, conf, j.float()), 1)[conf.view(-1) > conf_thres]

        if classes is not None:
            x = x[(x[:, 5:6] == torch.tensor(classes, device=x.device)).any(1)]

        n = x.shape[0]
        if not n:
            continue

        boxes, scores = x[:, :4], x[:, 4]
        i = torchvision.ops.nms(boxes, scores, iou_thres)
        output[xi] = x[i]
    return output

# Load YOLO model
yaml_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\cfg\deploy\yolov7.yaml"
weights_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\yolov7.pt"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model(cfg=yaml_path).to(device)
checkpoint = torch.load(weights_path, map_location=device)
model.load_state_dict(checkpoint['model'].state_dict(), strict=False)
model.eval()

# Folder with image sequence
image_folder = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data"
image_files = sorted([f for f in os.listdir(image_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])

# Iterate through images
for img_file in image_files:
    img_path = os.path.join(image_folder, img_file)
    img0 = cv2.imread(img_path)
    img_resized = cv2.resize(img0, (640, 640))
    img_normalized = img_resized / 255.0
    img = torch.from_numpy(img_normalized).float().permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        predictions = model(img)[0]
        detections = non_max_suppression(predictions, conf_thres=0.25, iou_thres=0.45, classes=[0, 1, 2])

    for det in detections:
        if len(det):
            det[:, :4] = scale_coords(img.shape[2:], det[:, :4], img0.shape).round()
            for *xyxy, conf, cls in reversed(det):
                c1, c2 = (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3]))
                color = (0, 255, 0)
                cv2.rectangle(img0, c1, c2, color, 2)
                label = f'{int(cls)} {conf:.2f}'
                cv2.putText(img0, label, (c1[0], c1[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    cv2.imshow('Detections', img0)
    key = cv2.waitKey(0)  # Press any key to go to the next image
    if key == 27:  # Exit if 'Esc' is pressed
        break

cv2.destroyAllWindows()


c:\Users\szakt\Desktop\DTU\venv\Lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
C:\Users\szakt\AppData\Local\Temp\ipykernel_6160\3143412441.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be l

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO
model = YOLO(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolo11x-seg.pt")

img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data\000000.png"

results = model.predict(source=img_path, classes = [0,1,2])


image 1/1 C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data\000000.png: 224x640 6 persons, 2 bicycles, 785.7ms
Speed: 0.0ms preprocess, 785.7ms inference, 16.1ms postprocess per image at shape (1, 3, 224, 640)


In [2]:
result = results[0]
boxes = result.boxes  # Boxes object for bounding box outputs
masks = result.masks  # Masks object for segmentation masks outputs
keypoints = result.keypoints  # Keypoints object for pose outputs
probs = result.probs  # Probs object for classification outputs
obb = result.obb  # Oriented boxes object for OBB outputs
result.show()  # display to screen
mask_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\temp.txt"
result.save_txt(mask_path)

In [ ]:
import numpy as np
import cv2
import json

def get_polygon_points(image, mask_file, output_json):
    """
    Given an image and a mask file with polygon coordinates,
    extracts all pixel coordinates inside the polygon(s) and saves them into a JSON file,
    including class labels and polygon coordinates for each polygon.

    Parameters:
    - image: NumPy array representing the image (used for resolution).
    - mask_file: Path to the mask text file.
    - output_json: Path to the output JSON file.

    Returns:
    - objects_with_points: A list of dictionaries containing class labels, polygon coordinates, and points inside the polygon.
    """
    img_height, img_width = image.shape[:2]
    
    # Step 1: Parse the mask file
    objects = []
    with open(mask_file, 'r') as file:
        for line in file:
            tokens = line.strip().split()
            if len(tokens) < 3:
                continue
            class_label = int(tokens[0])
            coords = list(map(float, tokens[1:]))
            if len(coords) % 2 != 0:
                continue
            polygon = [(coords[i], coords[i + 1]) for i in range(0, len(coords), 2)]
            objects.append({
                'class_label': class_label,
                'polygon': polygon
            })
    
    # Step 2: Convert normalized coordinates to pixel coordinates
    for obj in objects:
        polygon = obj['polygon']
        pixel_polygon = [(int(round(x * img_width)), int(round(y * img_height))) for x, y in polygon]
        obj['pixel_polygon'] = pixel_polygon
    
    # Initialize the list to store objects with their points
    objects_with_points = []

    # For each object, create a mask and extract points
    for obj in objects:
        class_label = obj['class_label']
        pixel_polygon = np.array([obj['pixel_polygon']], dtype=np.int32)
        # Create a mask for the current polygon
        mask = np.zeros((img_height, img_width), dtype=np.uint8)
        cv2.fillPoly(mask, pixel_polygon, color=1)
        # Extract points inside the polygon
        y_coords, x_coords = np.where(mask == 1)
        points_inside = list(zip(map(int, x_coords), map(int, y_coords)))
        # Store the data
        objects_with_points.append({
            'class_label': class_label,
            # 'polygon': obj['pixel_polygon'],
            'points': points_inside
        })  
    
    # Save to JSON file
    with open(output_json, 'w') as f:
        json.dump(objects_with_points, f)
    
    return objects_with_points


In [10]:
mask_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\temp.txt"
output_json = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\points.json"
img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data\000000.png"


import numpy as np
import cv2

image = cv2.imread(img_path)

points_with_labels = get_polygon_points(image, mask_path, output_json)

print(f"Total points extracted: {len(points_with_labels)}")


Total points extracted: 16


## Pointcloud clustering

In [ ]:
import open3d as o3d
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
pcl = o3d.io.read_point_cloud(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\test_cloud.ply")


# Downsample the point cloud
voxel_size = 0.5
pcl_ds = pcl.voxel_down_sample(voxel_size)
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

# K-Means function with WCSS calculation
def kmeans(X, k, max_iters=100, tol=1e-4):
    def initialize_centroids(X, k):
        np.random.seed(42)
        random_indices = np.random.choice(X.shape[0], k, replace=False)
        return X[random_indices]
    
    def assign_clusters(X, centroids):
        distances = np.sqrt(((X[:, np.newaxis] - centroids) ** 2).sum(axis=2))
        return np.argmin(distances, axis=1)
    
    def update_centroids(X, cluster_assignments, k):
        return np.array([X[cluster_assignments == i].mean(axis=0) for i in range(k)])
    
    centroids = initialize_centroids(X, k)
    for i in range(max_iters):
        cluster_assignments = assign_clusters(X, centroids)
        new_centroids = update_centroids(X, cluster_assignments, k)
        
        if np.linalg.norm(new_centroids - centroids) < tol:
            break
        centroids = new_centroids
    
    return centroids, cluster_assignments

# Function to calculate WCSS (Inertia)
def calculate_wcss(X, centroids, cluster_assignments):
    wcss = 0
    for i in range(len(centroids)):
        cluster_points = X[cluster_assignments == i]
        wcss += ((cluster_points - centroids[i]) ** 2).sum()
    return wcss

# Function to find optimal k using the Elbow Method
def elbow_method(X, max_k=10):
    wcss_values = []
    for k in range(1, max_k + 1):
        centroids, cluster_assignments = kmeans(X, k)
        wcss = calculate_wcss(X, centroids, cluster_assignments)
        wcss_values.append(wcss)
    
    # Plot WCSS vs k
    plt.plot(range(1, max_k + 1), wcss_values, marker='o')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('WCSS (Inertia)')
    plt.title('Elbow Method for Optimal k')
    plt.show()

    return wcss_values

# Assign random colors to clusters
def assign_colors(labels, n_clusters):
    np.random.seed(40)
    colors = np.random.rand(n_clusters, 3)  # Random colors for each cluster
    return np.array([colors[label] for label in labels])

# Generate some random 3D points (for demonstration)
X_3d = np.asarray(pcl_ds.points)

# Use the Elbow method to determine the optimal number of clusters
wcss_values = elbow_method(X_3d, max_k=10)

# From the plot, determine the optimal number of clusters (you can manually choose the elbow point)
optimal_k = 6  # Assume the elbow shows k=5

# Apply K-Means with the optimal number of clusters
centroids, labels = kmeans(X_3d, optimal_k)

# Assign colors to the clusters
cluster_colors = assign_colors(labels, optimal_k)

# Create Open3D PointCloud object
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(X_3d)  # Assign 3D points
point_cloud.colors = o3d.utility.Vector3dVector(cluster_colors)  # Assign colors based on clusters

# Visualize the point cloud with clusters using Open3D
o3d.visualization.draw_geometries([point_cloud])


C:\Users\szakt\AppData\Local\Temp\ipykernel_23044\1340798285.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

def cluster_point_cloud(point_cloud, num_clusters, voxel_size=0.5, max_iters=100, tol=1e-4):
    """
    Clusters a point cloud into the specified number of clusters using K-Means.

    Parameters:
    - point_cloud (open3d.geometry.PointCloud): The input point cloud.
    - num_clusters (int): The desired number of clusters.
    - voxel_size (float): Voxel size for downsampling. Default is 0.5.
    - max_iters (int): Maximum iterations for K-Means. Default is 100.
    - tol (float): Tolerance for convergence in K-Means. Default is 1e-4.

    Returns:
    - clustered_pcd (open3d.geometry.PointCloud): The clustered point cloud with colors assigned to each cluster.
    """

    # Downsample the point cloud
    pcl_ds = point_cloud.voxel_down_sample(voxel_size)

    # Convert Open3D point cloud to NumPy array
    X_3d = np.asarray(pcl_ds.points)

    # Standardize the data
    X_3d_mean = X_3d.mean(axis=0)
    X_3d_std = X_3d.std(axis=0)
    X_3d_standardized = (X_3d - X_3d_mean) / X_3d_std

    # K-Means clustering
    def kmeans(X, k, max_iters=100, tol=1e-4):
        def initialize_centroids(X, k):
            np.random.seed(42)
            random_indices = np.random.choice(X.shape[0], k, replace=False)
            return X[random_indices]

        def assign_clusters(X, centroids):
            distances = np.sqrt(((X[:, np.newaxis] - centroids) ** 2).sum(axis=2))
            return np.argmin(distances, axis=1)

        def update_centroids(X, cluster_assignments, k):
            return np.array([X[cluster_assignments == i].mean(axis=0) for i in range(k)])

        centroids = initialize_centroids(X, k)
        for i in range(max_iters):
            cluster_assignments = assign_clusters(X, centroids)
            new_centroids = update_centroids(X, cluster_assignments, k)

            if np.linalg.norm(new_centroids - centroids) < tol:
                break
            centroids = new_centroids

        return centroids, cluster_assignments

    # Apply K-Means
    centroids, labels = kmeans(X_3d_standardized, num_clusters, max_iters, tol)

    # Assign colors to clusters
    def assign_colors(labels, n_clusters):
        np.random.seed(40)
        colors = np.random.rand(n_clusters, 3)  # Random colors for each cluster
        return np.array([colors[label] for label in labels])

    cluster_colors = assign_colors(labels, num_clusters)

    # Create Open3D PointCloud object with cluster colors
    clustered_pcd = o3d.geometry.PointCloud()
    clustered_pcd.points = o3d.utility.Vector3dVector(X_3d)
    clustered_pcd.colors = o3d.utility.Vector3dVector(cluster_colors)

    return clustered_pcd

# Load your point cloud
pcl = o3d.io.read_point_cloud(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\test_cloud.ply")

# Define the number of clusters
num_clusters = 6

# Cluster the point cloud
clustered_pcd = cluster_point_cloud(pcl, num_clusters, voxel_size=0.5)

# Visualize the clustered point cloud
o3d.visualization.draw_geometries([clustered_pcd])


[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 


## DBSCAN

In [24]:
import json
import numpy as np
from sklearn.cluster import DBSCAN
from collections import defaultdict
import open3d as o3d

In [25]:
import cv2
import numpy as np
import json
import open3d as o3d
from image_processing import parse_calibration_data, create_seg_mask, RectImg2PC

def generate_segmented_point_cloud(
    left_image_path: str,
    right_image_path: str,
    color_image_path: str,
    calibration_file_path: str,
    seg_json_path: str,
    max_z: float = 20.0,
    target_class_labels: list = None,
    save_point_cloud: bool = False,
    save_path: str = 'segmented_point_cloud.npy',
    visualize: bool = False
) -> tuple:
    """
    Generates a segmented point cloud from stereo images and segmentation data.
    
    Returns:
    - point_cloud (np.ndarray): Array of shape (N, 3) or (N, 6) if colors are included.
    - mask (np.ndarray): Binary mask used for segmentation.
    """
    # Step 1: Load and convert images to grayscale
    rect_img1 = cv2.imread(left_image_path, cv2.IMREAD_GRAYSCALE)
    rect_img2 = cv2.imread(right_image_path, cv2.IMREAD_GRAYSCALE)
    
    if rect_img1 is None:
        raise FileNotFoundError(f"Left grayscale image not found at path: {left_image_path}")
    if rect_img2 is None:
        raise FileNotFoundError(f"Right grayscale image not found at path: {right_image_path}")
    
    # Step 2: Load the color image and convert to RGB
    color_image_bgr = cv2.imread(color_image_path)
    if color_image_bgr is None:
        raise FileNotFoundError(f"Color image not found at path: {color_image_path}")
    color_img = cv2.cvtColor(color_image_bgr, cv2.COLOR_BGR2RGB)
    
    # Step 3: Load and parse calibration data
    calibration = parse_calibration_data(calibration_file_path)
    if "P_rect_02" not in calibration or "P_rect_03" not in calibration:
        raise KeyError("Calibration data must contain 'P_rect_02' and 'P_rect_03'.")
    P_left = calibration["P_rect_02"]
    P_right = calibration["P_rect_03"]
    
    # Step 4: Create segmentation mask
    seg_mask = create_seg_mask(
        file_path=seg_json_path,
        image_shape=rect_img1.shape,
        target_class_labels=target_class_labels
    )
    
    # Step 5: Generate the point cloud using stereo images, calibration, and mask
    point_cloud = RectImg2PC(
        rect_img1=rect_img1,
        rect_img2=rect_img2,
        P_left=P_left,
        P_right=P_right,
        max_z=max_z,
        color_img=color_img,
        mask=seg_mask
    )
    
    # Step 6: Save the point cloud if required
    if save_point_cloud:
        np.save(save_path, point_cloud)
        print(f"Point cloud saved to {save_path}")
    
    # Step 7: Visualize the point cloud if required
    if visualize:
        if point_cloud.shape[1] < 6:
            raise ValueError("Point cloud does not contain color information (requires at least 6 columns).")
        
        # Separate XYZ and RGB
        xyz = point_cloud[:, :3]
        rgb = point_cloud[:, 3:] / 255.0  # Normalize RGB to [0, 1]
        
        # Create Open3D point cloud object
        o3d_pcd = o3d.geometry.PointCloud()
        o3d_pcd.points = o3d.utility.Vector3dVector(xyz)
        o3d_pcd.colors = o3d.utility.Vector3dVector(rgb)
        
        # Visualize
        o3d.visualization.draw_geometries([o3d_pcd], window_name="Segmented Point Cloud")
    
    return point_cloud, seg_mask


left_image_path = '../34759_final_project_rect/seq_01/image_02/data/000000.png'
right_image_path = '../34759_final_project_rect/seq_01/image_03/data/000000.png'
color_image_path = '../34759_final_project_rect/seq_01/image_02/data/000000.png'  # Assuming same as left
calibration_file_path = '../34759_final_project_rect/calib_cam_to_cam.txt'
seg_json_path = '../points.json'

point_cloud, seg_mask = generate_segmented_point_cloud(
    left_image_path=left_image_path,
    right_image_path=right_image_path,
    color_image_path=color_image_path,
    calibration_file_path=calibration_file_path,
    seg_json_path=seg_json_path,
    max_z=20.0,
    target_class_labels=None,
    save_point_cloud=False,
    save_path='segmented_point_cloud.npy',
    visualize=False  # Set to True if you want to visualize
)

In [22]:
def filter_json_to_pointcloud(json_file: str, mask: np.ndarray, output_json_file: str) -> list:
    """
    Filters the JSON data to include only points present in the point cloud based on the mask.
    
    Parameters:
    - json_file (str): Path to the input JSON file.
    - mask (np.ndarray): Binary mask used to generate the point cloud. Shape: (H, W).
    - output_json_file (str): Path to save the filtered JSON file.
    
    Returns:
    - filtered_data (list): The filtered JSON data.
    """
    # Load JSON data
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Extract (x, y) coordinates from the mask in the order of the point cloud
    ys, xs = np.where(mask)
    list_of_xy = list(zip(xs, ys))
    set_of_xy = set(list_of_xy)
    
    # Build a mapping from (x, y) to class_label from JSON
    xy_to_class = {}
    for obj in data:
        class_label = obj['class_label']
        for point in obj['points']:
            x, y = point
            xy_to_class[(x, y)] = class_label
    
    # Assign class labels to each point in the point cloud
    class_labels = []
    for (x, y) in list_of_xy:
        class_label = xy_to_class.get((x, y), -1)  # -1 for unknown/noise
        if class_label != -1:
            class_labels.append(class_label)
        else:
            # Optionally handle unknown classes
            class_labels.append(-1)
    
    # Verify consistency
    if len(class_labels) != point_cloud.shape[0]:
        print(f"Warning: Number of class labels ({len(class_labels)}) does not match number of points in point cloud ({point_cloud.shape[0]}).")
    
    # Reconstruct the filtered JSON structure
    # Since the point cloud is a flat list of points from the mask, it's difficult to map back to individual detections.
    # Instead, we'll aggregate class labels.
    
    # Alternatively, if you want to retain object-wise structure, more complex mapping is needed.
    # Here, we'll proceed with a flat structure.
    
    filtered_data = []
    for idx, class_label in enumerate(class_labels):
        filtered_data.append({
            'class_label': class_label,
            'point': list(point_cloud[idx])
        })
    
    # Save the filtered JSON data
    with open(output_json_file, 'w') as f:
        json.dump(filtered_data, f, indent=4)
    
    print(f"Filtered JSON saved to {output_json_file}")
    print(f"Total points in filtered JSON: {len(filtered_data)}")
    
    return filtered_data


In [23]:
import numpy as np

# Paths to your data
json_file = '../points.json'  # Path to the original JSON file
filtered_json_file = '../filtered_points.json'  # Path to save the filtered JSON

# Load the generated point cloud
point_cloud = np.asarray(point_cloud.points)

# Filter the JSON data to match the point cloud
# Filter the JSON data
filtered_data = filter_json_to_pointcloud(
    json_file=json_file,
    mask=seg_mask,
    output_json_file=filtered_json_file
)
# Optional: Verify the filtered data
print(f"First object in filtered JSON: {filtered_data[0]}")


IndexError: index 23355 is out of bounds for axis 0 with size 23355

In [ ]:
def assign_class_labels(filtered_json_file: str) -> list:
    """
    Assigns class labels to each point in the point cloud based on the filtered JSON data.
    
    Parameters:
    - filtered_json_file (str): Path to the filtered JSON file.
    
    Returns:
    - class_labels (list): List of class labels corresponding to each point in the point cloud.
    """
    with open(filtered_json_file, 'r') as f:
        data = json.load(f)
    
    class_labels = []
    for obj in data:
        class_labels.append(obj['class_label'])
    
    return class_labels

# Assign class labels
class_labels = assign_class_labels(filtered_json_file)


In [ ]:
from sklearn.cluster import DBSCAN
from collections import defaultdict

def cluster_point_cloud_by_class(point_cloud: np.ndarray, class_labels: list, eps: float = 0.5, min_samples: int = 10) -> dict:
    """
    Clusters the point cloud using DBSCAN for each class and computes centroids.
    
    Parameters:
    - point_cloud (np.ndarray): Array of shape (N, 3) representing the point cloud.
    - class_labels (list): List of class labels for each point.
    - eps (float): DBSCAN epsilon parameter.
    - min_samples (int): DBSCAN min_samples parameter.
    
    Returns:
    - centroids_per_class (dict): Dictionary mapping class_label to list of centroids.
    """
    unique_classes = set(class_labels)
    unique_classes.discard(-1)  # Remove noise class if present
    centroids_per_class = {}
    
    for class_label in unique_classes:
        # Extract points belonging to the current class
        class_points = point_cloud[np.array(class_labels) == class_label]
        print(f"Clustering Class {class_label} with {class_points.shape[0]} points.")
        
        # Apply DBSCAN
        db = DBSCAN(eps=eps, min_samples=min_samples)
        db.fit(class_points)
        labels = db.labels_
        
        # Number of clusters, ignoring noise (-1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"Class {class_label}: Estimated number of clusters: {n_clusters}")
        
        # Organize points by cluster label
        clusters = defaultdict(list)
        for point, label in zip(class_points, labels):
            if label == -1:
                continue  # Skip noise
            clusters[label].append(point)
        
        # Compute centroids for each cluster
        centroids = []
        for cluster_label, points in clusters.items():
            centroid = np.mean(points, axis=0)
            centroids.append(centroid)
            print(f"Class {class_label}, Cluster {cluster_label}: Centroid at {centroid}")
        
        centroids_per_class[class_label] = centroids
    
    return centroids_per_class

# Cluster the point cloud and compute centroids
centroids_per_class = cluster_point_cloud_by_class(
    point_cloud=point_cloud, 
    class_labels=class_labels, 
    eps=0.5, 
    min_samples=10
)

In [4]:
# Path to your JSON file
json_file = r'C:\Users\szakt\Desktop\DTU\Perception\FinalProject\filtered_points.json'

# Load JSON data
with open(json_file, 'r') as f:
    data = json.load(f)

# # Load your point cloud data
# point_cloud_file = r'C:\Users\szakt\Desktop\DTU\Perception\FinalProject\filtered_point_cloud.ply'

# # Load point cloud data
# pc = o3d.io.read_point_cloud(point_cloud_file)

# point_cloud = np.asarray(pc.points)

In [5]:
# Initialize lists to store data
class_labels = []
points_3d = []

# Counter to track the index in the point cloud
point_index = 0

for obj in data:
    class_label = obj['class_label']
    pixel_points = obj['points']  # List of [x, y] pixel coordinates

    num_points = len(pixel_points)
    # Extract the corresponding 3D points from the point cloud
    corresponding_points = point_cloud[point_index:point_index + num_points]

    # Append class labels and 3D points
    class_labels.extend([class_label] * num_points)
    points_3d.extend(corresponding_points)

    # Update the point index
    point_index += num_points


In [6]:
assert point_index == point_cloud.shape[0], "Mismatch between JSON data and point cloud data."


In [7]:
class_labels = np.array(class_labels)  # Shape: (N,)
points_3d = np.array(points_3d)        # Shape: (N, 3)


In [8]:
# Get unique class labels
unique_classes = np.unique(class_labels)

# Dictionary to hold points per class
points_per_class = {class_label: points_3d[class_labels == class_label] for class_label in unique_classes}


In [17]:
# Parameters for DBSCAN (adjust these as needed)
eps = 0.5      # Maximum distance between two samples
min_samples = 100  # Minimum number of points to form a cluster

# Dictionary to store centroids per class
centroids_per_class = {}

for class_label in unique_classes:
    # Get the points for the current class
    class_points = points_per_class[class_label]

    # Run DBSCAN clustering
    db = DBSCAN(eps=eps, min_samples=min_samples)
    db.fit(class_points)

    # Get cluster labels (-1 denotes noise)
    cluster_labels = db.labels_

    # Number of clusters (excluding noise)
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    print(f'Class {class_label}: Estimated number of clusters: {n_clusters}')

    # Compute centroids for each cluster
    clusters = defaultdict(list)
    for point, cluster_label in zip(class_points, cluster_labels):
        if cluster_label != -1:
            clusters[cluster_label].append(point)

    # Calculate centroids
    centroids = {}
    for cluster_label, points in clusters.items():
        points_array = np.array(points)
        centroid = points_array.mean(axis=0)
        centroids[cluster_label] = centroid
        print(f'Class {class_label}, Cluster {cluster_label}: Centroid at {centroid}')

    # Store centroids for the current class
    centroids_per_class[class_label] = centroids


Class 0: Estimated number of clusters: 5
Class 0, Cluster 0: Centroid at [ 1.03454628 -0.32953592  5.64171326]
Class 0, Cluster 1: Centroid at [ 3.02358106  0.39346272 19.6358199 ]
Class 0, Cluster 2: Centroid at [ 3.04108056 -0.73587073  7.79407009]
Class 0, Cluster 3: Centroid at [ 2.49526234 -0.40020248  5.05684358]
Class 0, Cluster 4: Centroid at [ 6.027597   -0.94889583 12.89951414]
Class 1: Estimated number of clusters: 2
Class 1, Cluster 0: Centroid at [ 1.12127375 -1.03220706  5.66314368]
Class 1, Cluster 1: Centroid at [ 2.93036401 -1.09484862  7.3131339 ]


In [18]:
import open3d as o3d
import matplotlib.pyplot as plt

# Prepare colors for visualization
colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, 20))

visualization_list = []

for class_label in unique_classes:
    class_points = points_per_class[class_label]
    db = DBSCAN(eps=eps, min_samples=min_samples)
    db.fit(class_points)
    cluster_labels = db.labels_

    clusters = defaultdict(list)
    for point, cluster_label in zip(class_points, cluster_labels):
        if cluster_label != -1:
            clusters[cluster_label].append(point)

    # Visualize clusters
    for cluster_label, points in clusters.items():
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        color = colors[int(class_label) % 20][:3]  # Use class_label to select color
        pcd.paint_uniform_color(color)
        visualization_list.append(pcd)

        # Add centroid as a small sphere
        centroid = centroids_per_class[class_label][cluster_label]
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.05)
        sphere.translate(centroid)
        sphere.paint_uniform_color([0, 0, 0])  # Black color for centroids
        visualization_list.append(sphere)

o3d.visualization.draw_geometries(visualization_list)


C:\Users\szakt\AppData\Local\Temp\ipykernel_24184\2233442088.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, 20))
